In [7]:
# @title Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from evaluate import load
from transformers import pipeline
import os

In [8]:
# @title Data Preparation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

spacy_de = spacy.load("de_core_news_sm")
spacy_en = spacy.load("en_core_web_sm")

def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)]
def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]

UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3
special_tokens = ['<unk>', '<pad>', '<sos>', '<eos>']

def build_vocab(sentences, tokenizer, lang):
    counter = Counter()
    for sentence in sentences:
        tokens = tokenizer(sentence[lang])
        counter.update(tokens)
    
    vocab = {word: i for i, word in enumerate(special_tokens)}
    for word, count in counter.items():
        if count >= 2:
            vocab[word] = len(vocab)
    return vocab

print("Loading dataset...")
data_files = {
    "train": "datasets/multi30k/train.csv",
    "validation": "datasets/multi30k/validation.csv",
    "test": "datasets/multi30k/test.csv"
}

dataset = load_dataset("csv", data_files=data_files)
train_data = dataset["train"]

print("Building vocabularies...")
vocab_de = build_vocab(train_data, tokenize_de, 'de')
vocab_en = build_vocab(train_data, tokenize_en, 'en')
print(f"German vocab size: {len(vocab_de)}")
print(f"English vocab size: {len(vocab_en)}")

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for item in batch:
        src_tokens = [vocab_de.get(tok, UNK_IDX) for tok in tokenize_de(item['de'])]
        trg_tokens = [vocab_en.get(tok, UNK_IDX) for tok in tokenize_en(item['en'])]
        src_batch.append(torch.tensor([SOS_IDX] + src_tokens + [EOS_IDX]))
        trg_batch.append(torch.tensor([SOS_IDX] + trg_tokens + [EOS_IDX]))
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX)
    return {'src': src_batch, 'trg': trg_batch}

BATCH_SIZE = 128

train_loader = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
validation_loader = DataLoader(dataset['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset['test'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print("Data loading complete.")

Loading dataset...
Building vocabularies...
German vocab size: 8014
English vocab size: 6191
Data loading complete.


In [9]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional=True)
        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs.permute(1, 0, 2)), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(enc_hid_dim * 2 + emb_dim, dec_hid_dim)
        self.fc_out = nn.Linear(enc_hid_dim * 2 + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs.permute(1, 0, 2))
        rnn_input = torch.cat((embedded, weighted.permute(1, 0, 2)), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        output = output.squeeze(0)
        embedded = embedded.squeeze(0)
        weighted = weighted.squeeze(1)
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden.squeeze(0)
    
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
        return outputs

In [10]:
def train_step(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for batch in enumerate(iterator):
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. The Pre-trained Transformer Baseline (Explicit Loading)
print("Loading Transformer Baseline...")
transformer_name = "Helsinki-NLP/opus-mt-de-en"

# to gpu
baseline_tokenizer = AutoTokenizer.from_pretrained(transformer_name)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(transformer_name, use_safetensors=True).to(device)

# Example Inference
german_text = "Ein Hund rennt über das Gras."

# Convert text to tensor tokens, push to GPU
inputs = baseline_tokenizer(german_text, return_tensors="pt").to(device)

# Generate translation tokens
outputs = baseline_model.generate(**inputs)

# Decode tokens back to english text
transformer_output = baseline_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Transformer Translation: {transformer_output}")

# 2. Evaluation Suite
bleu = load("sacrebleu")
meteor = load("meteor")
chrf = load("chrf")
bertscore = load("bertscore")

def evaluate_models(predictions, references):
    bleu_score = bleu.compute(predictions=predictions, references=references)
    meteor_score = meteor.compute(predictions=predictions, references=references)
    chrf_score = chrf.compute(predictions=predictions, references=references)
    bert_s = bertscore.compute(predictions=predictions, references=references, lang="en")
    
    print(f"BLEU: {bleu_score['score']:.2f}")
    print(f"METEOR: {meteor_score['meteor']:.2f}")
    print(f"ChrF: {chrf_score['score']:.2f}")
    print(f"BERTScore (F1 mean): {sum(bert_s['f1'])/len(bert_s['f1']):.2f}")

Loading Transformer Baseline...


Loading weights: 100%|██████████| 256/256 [00:00<00:00, 3632.27it/s]


Transformer Translation: A dog runs over the grass.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\MUSTAFA\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MUSTAFA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\MUSTAFA\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [13]:
# Hyperparameters
INPUT_DIM = len(vocab_de)
OUTPUT_DIM = len(vocab_en)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
ENC_HID_DIM = 512
DEC_HID_DIM = 512
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
N_EPOCHS = 10
CLIP = 1.0

# Initialize Model
attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)

model = Seq2Seq(enc, dec, device).to(device)
optimizer = optim.Adam(model.parameters())
# We don't want the model learning to predict <pad>
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
# Run Training
best_valid_loss = float('inf')
print("Starting training...")
for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss / len(train_loader)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

print("Training complete. You can now pass test_loader through the model and evaluate with the Transformer baseline!")

Starting training...
Epoch: 01 | Train Loss: 4.251
Epoch: 02 | Train Loss: 3.072
Epoch: 03 | Train Loss: 2.633
Epoch: 04 | Train Loss: 2.340
Epoch: 05 | Train Loss: 2.130
Epoch: 06 | Train Loss: 1.960
Epoch: 07 | Train Loss: 1.869
Epoch: 08 | Train Loss: 1.774
Epoch: 09 | Train Loss: 1.702
Epoch: 10 | Train Loss: 1.608
Training complete. You can now pass test_loader through the model and evaluate with the Transformer baseline!


In [14]:
# --- EVALUATION SCRIPT ---
print("\nTranslating Test Set with Custom Model...")

custom_predictions = []
references = []

model.eval() # Turn off dropout
with torch.no_grad(): # Turn off gradient tracking for faster inference
    for batch in test_loader:
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)
        
        # Turn off teacher forcing for evaluation!
        output = model(src, trg, teacher_forcing_ratio=0)
        
        # output shape is [trg_len, batch_size, output_dim]
        # Get the highest probability token for each word
        top1 = output.argmax(2) 
        
        # We need to process batch by batch, sentence by sentence
        for i in range(top1.shape[1]): # Loop over batch_size
            pred_tokens = top1[1:, i] # Skip <sos> token
            ref_tokens = trg[1:, i]
            
            # Convert indices back to strings, stopping at <eos> or <pad>
            pred_words = []
            for token in pred_tokens:
                if token.item() == EOS_IDX or token.item() == PAD_IDX:
                    break
                # Find the word matching the index in the English vocab
                word = list(vocab_en.keys())[list(vocab_en.values()).index(token.item())]
                pred_words.append(word)
                
            ref_words = []
            for token in ref_tokens:
                if token.item() == EOS_IDX or token.item() == PAD_IDX:
                    break
                word = list(vocab_en.keys())[list(vocab_en.values()).index(token.item())]
                ref_words.append(word)
            
            custom_predictions.append(" ".join(pred_words))
            references.append(" ".join(ref_words))

print("Custom Model Evaluation:")
evaluate_models(custom_predictions, references)

# ---------------------------------------------------------
print("\nTranslating Test Set with Transformer Baseline...")
source_sentences = []
for batch in test_loader:
    for i in range(batch['src'].shape[1]):
        src_tokens = batch['src'][1:, i]
        src_words = []
        for token in src_tokens:
            if token.item() == EOS_IDX or token.item() == PAD_IDX:
                break
            word = list(vocab_de.keys())[list(vocab_de.values()).index(token.item())]
            src_words.append(word)
        source_sentences.append(" ".join(src_words))

transformer_predictions = []
for sentence in source_sentences:
    inputs = baseline_tokenizer(sentence, return_tensors="pt").to(device)
    outputs = baseline_model.generate(**inputs)
    translation = baseline_tokenizer.decode(outputs[0], skip_special_tokens=True)
    transformer_predictions.append(translation)

print("Transformer Baseline Evaluation:")
evaluate_models(transformer_predictions, references)


Translating Test Set with Custom Model...


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Custom Model Evaluation:


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3099.48it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BLEU: 30.34
METEOR: 0.61
ChrF: 49.90
BERTScore (F1 mean): 0.93

Translating Test Set with Transformer Baseline...
Transformer Baseline Evaluation:
BLEU: 33.26
METEOR: 0.66
ChrF: 55.14
BERTScore (F1 mean): 0.94
